# Chapter 19: Production Optimization Theory and Methods

This notebook covers the theoretical foundations of production optimization,
including:
- Building a simple constrained process (separator → compressor → export)
- Implementing golden section search manually for maximum throughput
- Comparing manual results with NeqSim's `ProcessOptimizationEngine`
- Plotting the objective function landscape
- Sensitivity tornado plot for constraint impacts

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 19.1 Build a Constrained Gas Export Process

We define a minimal process where the feed flows through a separator, a compressor,
and out to export. Equipment is auto-sized to establish design constraints.

In [2]:
from neqsim import jneqsim

# Lean gas fluid
fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 20.0, 60.0)
fluid.addComponent("nitrogen", 0.02)
fluid.addComponent("CO2", 0.01)
fluid.addComponent("methane", 0.88)
fluid.addComponent("ethane", 0.05)
fluid.addComponent("propane", 0.03)
fluid.addComponent("n-butane", 0.01)
fluid.setMixingRule("classic")
fluid.setMultiPhaseCheck(True)

design_rate_kg_hr = 45000.0

feed = jneqsim.process.equipment.stream.Stream("Feed", fluid)
feed.setFlowRate(design_rate_kg_hr, "kg/hr")
feed.setTemperature(20.0, "C")
feed.setPressure(60.0, "bara")

sep = jneqsim.process.equipment.separator.Separator("Separator", feed)

comp = jneqsim.process.equipment.compressor.Compressor("Compressor", sep.getGasOutStream())
comp.setOutletPressure(150.0)

cooler = jneqsim.process.equipment.heatexchanger.Cooler("Cooler", comp.getOutletStream())
cooler.setOutTemperature(273.15 + 30.0)

process = jneqsim.process.processmodel.ProcessSystem()
process.add(feed)
process.add(sep)
process.add(comp)
process.add(cooler)
process.run()

# Auto-size with 1.2 factor
sep.autoSize(1.2)
comp.autoSize(1.2)
process.run()

print(f"Design rate: {design_rate_kg_hr:.0f} kg/hr")
print(f"Compressor power at design: {comp.getPower('kW'):.1f} kW")
print(f"Separator auto-sized: {sep.isAutoSized()}")
print(f"Compressor auto-sized: {comp.isAutoSized()}")

Design rate: 45000 kg/hr
Compressor power at design: 1934.2 kW
Separator auto-sized: True
Compressor auto-sized: True


## 19.2 Objective Function Landscape

We sweep feed rate and record the production rate (= export gas rate) alongside
the maximum equipment utilization. The feasible region is where utilization < 100%.

In [3]:
flow_rates = np.linspace(15000, 75000, 25)
export_rates = []
max_utils = []
comp_powers = []

for rate in flow_rates:
    feed.setFlowRate(float(rate), "kg/hr")
    try:
        process.run()
        export_rate = cooler.getOutletStream().getFlowRate("kg/hr")
        export_rates.append(export_rate)
        comp_powers.append(comp.getPower("kW"))

        # Get max utilization
        bn = process.findBottleneck()
        if bn.hasBottleneck():
            max_utils.append(bn.getUtilizationPercent())
        else:
            max_utils.append(0.0)
    except Exception:
        export_rates.append(float('nan'))
        max_utils.append(float('nan'))
        comp_powers.append(float('nan'))

# Reset
feed.setFlowRate(design_rate_kg_hr, "kg/hr")
process.run()

# Convert to numpy
export_rates = np.array(export_rates)
max_utils = np.array(max_utils)
comp_powers = np.array(comp_powers)

In [4]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Export rate vs feed rate
ax1.plot(flow_rates/1000, export_rates/1000, 'b-o', linewidth=2, markersize=4, label='Export Rate')
ax1.set_ylabel('Export Gas Rate (t/hr)', fontsize=12)
ax1.set_title('Objective Function: Production Rate vs Feed Rate', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)

# Utilization vs feed rate (with feasible/infeasible shading)
ax2.plot(flow_rates/1000, max_utils, 'r-s', linewidth=2, markersize=4, label='Max Utilization')
ax2.axhline(y=100, color='red', linestyle='--', linewidth=1.5, label='Capacity Limit (100%)')
ax2.fill_between(flow_rates/1000, 0, max_utils,
                 where=(max_utils <= 100), alpha=0.15, color='green', label='Feasible Region')
ax2.fill_between(flow_rates/1000, 0, max_utils,
                 where=(max_utils > 100), alpha=0.15, color='red', label='Infeasible Region')
ax2.set_xlabel('Feed Rate (t/hr)', fontsize=12)
ax2.set_ylabel('Max Equipment Utilization (%)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch19_objective_function_landscape.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_37604\3771738393.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 19.3 Manual Golden Section Search

The golden section method narrows the search interval using the golden ratio
$\phi = (\sqrt{5}-1)/2 \approx 0.618$. We maximize the largest feasible rate.

In [5]:
def is_feasible(rate_kg_hr):
    """Check if the process is feasible at the given feed rate."""
    feed.setFlowRate(float(rate_kg_hr), "kg/hr")
    try:
        process.run()
        return not process.isAnyEquipmentOverloaded()
    except Exception:
        return False

# Golden section search for max feasible rate
phi = (np.sqrt(5) - 1) / 2  # golden ratio
a, b = 15000.0, 80000.0
tol_gs = 200.0  # kg/hr
gs_log = []

for i in range(40):
    if (b - a) < tol_gs:
        break
    x1 = b - phi * (b - a)
    x2 = a + phi * (b - a)

    f1 = is_feasible(x1)
    f2 = is_feasible(x2)

    gs_log.append({'iter': i+1, 'a': a, 'b': b, 'x1': x1, 'x2': x2,
                   'f1': f1, 'f2': f2})

    if f2:
        # x2 is feasible — search higher
        a = x1
    else:
        # x2 is infeasible — search lower
        b = x2

gs_max_rate = a

# Reset
feed.setFlowRate(design_rate_kg_hr, "kg/hr")
process.run()

print(f"Golden section max feasible rate: {gs_max_rate:.0f} kg/hr")
print(f"Iterations: {len(gs_log)}")
print(f"Headroom over design: {(gs_max_rate / design_rate_kg_hr - 1)*100:.1f}%")

Golden section max feasible rate: 42345 kg/hr
Iterations: 13
Headroom over design: -5.9%


## 19.4 Comparison with ProcessOptimizationEngine

We compare the manual golden section result with NeqSim's built-in optimizer.

In [6]:
ProcessOptimizationEngine = jneqsim.process.util.optimizer.ProcessOptimizationEngine

engine = ProcessOptimizationEngine(process)
engine.setFeedStreamName("Feed")
engine.setTolerance(0.001)
engine.setMaxIterations(60)
engine.setSearchAlgorithm(ProcessOptimizationEngine.SearchAlgorithm.GOLDEN_SECTION)

result_engine = engine.findMaximumThroughput(60.0, 120.0, 10000.0, 100000.0)

print(f"Manual golden section: {gs_max_rate:.0f} kg/hr")
print(f"NeqSim engine result:  {result_engine.getOptimalValue():.0f} kg/hr")
print(f"Difference:            {abs(gs_max_rate - result_engine.getOptimalValue()):.0f} kg/hr")
print(f"Engine iterations:     {result_engine.getOptimalValue()}")

# Reset
feed.setFlowRate(design_rate_kg_hr, "kg/hr")
process.run()

Manual golden section: 42345 kg/hr
NeqSim engine result:  10000 kg/hr
Difference:            32345 kg/hr
Engine iterations:     10000.000318195875


## 19.5 Sensitivity Tornado Plot

We vary each equipment's capacity by ±20% and measure the impact on maximum
throughput. This reveals which constraints have the greatest leverage.

In [7]:
def find_max_rate_binary(process, feed, low=15000, high=80000, tol=500):
    """Quick binary search for max feasible rate."""
    for _ in range(30):
        mid = (low + high) / 2.0
        feed.setFlowRate(float(mid), "kg/hr")
        try:
            process.run()
            if not process.isAnyEquipmentOverloaded():
                low = mid
            else:
                high = mid
        except Exception:
            high = mid
        if (high - low) < tol:
            break
    return low

# Baseline max rate
base_max = find_max_rate_binary(process, feed)
feed.setFlowRate(design_rate_kg_hr, "kg/hr")
process.run()

print(f"Baseline max rate: {base_max:.0f} kg/hr")

# We'll test sensitivity by re-auto-sizing with different factors
# Higher factor = more capacity = higher max rate
sensitivity_results = []

for equip_name, equip_obj in [("Separator", sep), ("Compressor", comp)]:
    for label, factor in [("- 20%", 1.0), ("+20%", 1.44)]:  # 1.0=no margin, 1.44=1.2*1.2
        # Re-auto-size this equipment
        feed.setFlowRate(design_rate_kg_hr, "kg/hr")
        process.run()
        equip_obj.autoSize(float(factor))
        process.run()

        max_rate = find_max_rate_binary(process, feed)

        sensitivity_results.append({
            'equipment': equip_name,
            'case': label,
            'factor': factor,
            'max_rate': max_rate
        })

        # Restore to baseline sizing
        feed.setFlowRate(design_rate_kg_hr, "kg/hr")
        process.run()
        equip_obj.autoSize(1.2)
        process.run()

print("\nSensitivity Results:")
for r in sensitivity_results:
    delta = r['max_rate'] - base_max
    print(f"  {r['equipment']} {r['case']}: {r['max_rate']:.0f} kg/hr (delta: {delta:+.0f})")

Baseline max rate: 42422 kg/hr



Sensitivity Results:


  Separator - 20%: 42422 kg/hr (delta: +0)
  Separator +20%: 42422 kg/hr (delta: +0)
  Compressor - 20%: 44961 kg/hr (delta: +2539)
  Compressor +20%: 46230 kg/hr (delta: +3809)


In [8]:
# Build tornado data
equip_names = list(set(r['equipment'] for r in sensitivity_results))
low_vals = []
high_vals = []

for ename in equip_names:
    entries = [r for r in sensitivity_results if r['equipment'] == ename]
    rates = [r['max_rate'] for r in entries]
    low_vals.append(min(rates) - base_max)
    high_vals.append(max(rates) - base_max)

# Sort by total swing
swings = [h - l for h, l in zip(high_vals, low_vals)]
order = np.argsort(swings)[::-1]
equip_names = [equip_names[i] for i in order]
low_vals = [low_vals[i] for i in order]
high_vals = [high_vals[i] for i in order]

fig, ax = plt.subplots(figsize=(10, 4))
y_pos = np.arange(len(equip_names))

ax.barh(y_pos, high_vals, left=0, color='steelblue', edgecolor='black',
        label='Capacity +20%', height=0.4)
ax.barh(y_pos, low_vals, left=0, color='coral', edgecolor='black',
        label='Capacity -20%', height=0.4)
ax.axvline(x=0, color='black', linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(equip_names, fontsize=11)
ax.set_xlabel(f'Change in Max Throughput (kg/hr) from baseline {base_max:.0f}', fontsize=11)
ax.set_title('Sensitivity Tornado: Equipment Capacity Impact on Max Production', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch19_sensitivity_tornado.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_37604\2457836315.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

Key theoretical concepts:

1. **Production optimization** is a constrained maximization: maximize throughput subject to equipment limits
2. **Golden section search** converges faster than binary search for smooth, unimodal objectives
3. The **objective function landscape** shows the linear export-rate region and the constraint boundary
4. **Sensitivity analysis** (tornado plot) reveals which equipment's capacity has the most leverage
5. NeqSim's `ProcessOptimizationEngine` automates these algorithms with built-in constraint enforcement